# PDF Input Non-Thinking Quickstart for Qwen3.5-27B on vLLM

この Notebook は、英語論文 PDF の **全ページ** を対象にして、vLLM の Qwen3 reasoning parser を使ったまま、**`enable_thinking=False` で non-thinking モード**にして text / image の両方を試すものです。


## 1. 前提
- Docker で vLLM API サーバが起動している
- `OPENAI_BASE_URL` を必要に応じて設定する
- PDF サンプルは `papers/` に配置済み


In [1]:
import os
from pathlib import Path
from openai import OpenAI

BASE_URL = os.environ.get('OPENAI_BASE_URL', 'http://127.0.0.1:8000/v1')
PAPERS = Path('../papers')
print('Using base_url =', BASE_URL)
print('Papers:', sorted(p.name for p in PAPERS.glob('*.pdf')))
client = OpenAI(api_key='EMPTY', base_url=BASE_URL, timeout=3600)
client


Using base_url = http://127.0.0.1:30010/v1
Papers: ['attention_is_all_you_need.pdf', 'deep_residual_learning.pdf']


## 2. PDF 全ページ text の non-thinking 要約


In [2]:
from pypdf import PdfReader

pdf_path = PAPERS / 'attention_is_all_you_need.pdf'
reader = PdfReader(str(pdf_path))
text = '\n'.join(page.extract_text() or '' for page in reader.pages)
print('page_count =', len(reader.pages))
print('text_chars =', len(text))


page_count = 15
text_chars = 39629


In [3]:
resp = client.chat.completions.create(
    model='Qwen/Qwen3.5-27B',
    messages=[
        {'role': 'system', 'content': 'You are reading a full English deep learning paper extracted from PDF text.'},
        {'role': 'user', 'content': 'Summarize the paper in Japanese with sections for problem, method, and key ideas.\n\n' + text[:60000]},
    ],
    extra_body={
        'chat_template_kwargs': {
            'enable_thinking': False
        }
    },
    max_tokens=512,
)
resp


ChatCompletion(id='chatcmpl-aa99fec9be6237bf', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='ご提示いただいた論文「Attention Is All You Need」（Transformer）の要約を、問題、手法、主要なアイデアの 3 つのセクションに分けて日本語で記述します。\n\n---\n\n# 論文要約：Attention Is All You Need (Transformer)\n\n### 1. 問題 (Problem)\n従来のシーケンス変換モデル（機械翻訳や言語モデルなど）の主流であった**リカレントニューラルネットワーク（RNN）**や**畳み込みニューラルネットワーク（CNN）**には、以下のような根本的な課題がありました。\n\n*   **並列化の困難さ**: RNN は計算を時間ステップごとに逐次的に行うため、トレーニング例内の計算を並列化できません。これは特に長いシーケンスにおいてボトルネックとなり、メモリ制約によりバッチ処理も制限されます。\n*   **長距離依存関係の学習難易度**: CNN や RNN では、入力シーケンス内の離れた位置の要素同士を関連付けるために、多くの層を通過させる必要があり（経路長が長い）、勾配の消失や伝播の遅れが発生しやすく、長距離の依存関係を学習するのが困難でした。\n*   **計算コスト**: 高品質なモデルを得るには、既存の最良モデル（アンサンブルなど）に比べて膨大なトレーニング時間と計算リソースが必要でした。\n\n### 2. 手法 (Method)\n本論文では、再帰（RNN）や畳み込み（CNN）を完全に排除し、**アテンション機構（Attention Mechanism）のみ**に基づいた新しいネットワークアーキテクチャ「**Transformer**」を提案しています。\n\n*   **アーキテクチャ**: エンコーダーとデコーダーのスタック構造を採用。各層は「マルチヘッド・セルフアテンション」と「位置ごとの全結合フィードフォワードネットワーク」の 2 つのサブレイヤーで構成され、

## 3. PDF 全ページ image の non-thinking 要約


In [4]:
import fitz  # pymupdf

pdf_path = PAPERS / 'deep_residual_learning.pdf'
doc = fitz.open(pdf_path)
out_dir = PAPERS / 'rendered' / 'deep_residual_learning_all_pages_vllm_no_thinking'
out_dir.mkdir(parents=True, exist_ok=True)
image_paths = []
for i in range(len(doc)):
    page = doc.load_page(i)
    pix = page.get_pixmap(matrix=fitz.Matrix(1.5, 1.5))
    img_path = out_dir / f'deep_residual_learning_page_{i+1:02d}.png'
    pix.save(img_path)
    image_paths.append(img_path)
print('rendered_pages =', len(image_paths))


rendered_pages = 12


In [5]:
import base64
import mimetypes

content = [{'type': 'text', 'text': 'These are rendered pages from an English deep learning paper PDF. Summarize the paper in Japanese and mention the overall topic and architecture.'}]
for img_path in image_paths:
    mime = mimetypes.guess_type(img_path.name)[0] or 'image/png'
    image_url = 'data:' + mime + ';base64,' + base64.b64encode(img_path.read_bytes()).decode('utf-8')
    content.append({'type': 'image_url', 'image_url': {'url': image_url}})

resp = client.chat.completions.create(
    model='Qwen/Qwen3.5-27B',
    messages=[{'role': 'user', 'content': content}],
    extra_body={
        'chat_template_kwargs': {
            'enable_thinking': False
        }
    },
    max_tokens=512,
)
resp


ChatCompletion(id='chatcmpl-b5721b1cc9d709cc', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='この論文は、画像認識のための「深層残差学習 (Deep Residual Learning)」という新しいフレームワークを提案しています。\n\n**トピック:**\n従来の畳み込みニューラルネットワーク (CNN) を深くすると、精度が飽和し、逆に低下する「退化 (degradation)」の問題が発生します。この論文は、その原因が過学習ではなく、深いネットワークの最適化の難しさにあることを指摘し、これを解決するための手法として残差学習を導入します。\n\n**アーキテクチャ:**\n提案手法の核心は、各層が入力 x から直接出力 y を学習するのではなく、入力 x と出力 y の差（残差）F(x) = y - x を学習するようにネットワークを再設計することです。これにより、ネットワークは恒等写像 (identity mapping) を容易に学習できるようになり、非常に深いネットワークでも効果的に訓練できるようになります。この仕組みを実現する基本的な構成要素が「残差ブロック」であり、入力と出力を「ショートカット接続」で直接つなぐのが特徴です。\n\nこのアーキテクチャにより、152層や1000層を超える非常に深いネットワークを構築・訓練することが可能になり、ImageNetやCOCOなどのベンチマークで当時の最高精度を達成しました。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning=None), stop_reason=None, token_ids=None)], created=1774942463, model='Qwen/Qwen3.5-27B', object='chat.completion', service_tier=None, system_fingerprint=None, usage